In [87]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime
from scipy import interpolate

In [88]:
start_date = '1984-01-01'
end_date = '2025-12-31'

# Create a date range for all months in the period
date_range = pd.date_range(start=start_date, end=end_date, freq='QS')
master_df = pd.DataFrame({'Date': date_range})

print(f"Created quarterly DataFrame with {len(monthly_df)} records from {monthly_df['Date'].min().strftime('%Y-%m-%d')} to {monthly_df['Date'].max().strftime('%Y-%m-%d')}")

Created quarterly DataFrame with 168 records from 1984-01-01 to 2025-10-01


In [89]:
House_Index = pd.read_csv(os.path.join('New_Data', 'DC_House_Price_Index.csv'))

# Convert the 'observation_date' column to datetime format

House_Index['observation_date'] = pd.to_datetime(House_Index['observation_date'])
# Rename observation_date to Date
House_Index.rename(columns={'observation_date': 'Date'}, inplace=True)
# Convert the 'Date' column to datetime format
House_Index['Date'] = pd.to_datetime(House_Index['Date'])
# Rename House Index Column
House_Index.rename(columns={'ATNHPIUS47894Q': 'House_Index'}, inplace=True)

# Merge the House_Index dolumn with the monthly_df DataFrame
master_df = pd.merge(master_df, House_Index[['Date', 'House_Index']], on='Date', how='left')

print(master_df.head())

        Date  House_Index
0 1984-01-01        57.37
1 1984-04-01        59.74
2 1984-07-01        60.50
3 1984-10-01        60.87
4 1985-01-01        61.61


In [90]:
cpi_df = pd.read_csv(os.path.join('New_Data', 'DMV_CPI.csv'))

# Convert from wide to long format
cpi_long = pd.melt(
    cpi_df,
    id_vars=['Year'],
    value_vars=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
    var_name='Month',
    value_name='CPI'
)

# Map month names to numbers
month_map = {'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6, 
             'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12}
cpi_long['Month'] = cpi_long['Month'].map(month_map)

# Create a proper Date column (always first of the month)
cpi_long['Date'] = pd.to_datetime(dict(year=cpi_long['Year'], month=cpi_long['Month'], day=1))

# Keep only Date and CPI
cpi_clean = cpi_long[['Date', 'CPI']].sort_values('Date').reset_index(drop=True)

print(cpi_clean.head())

# Merge the CPI data with the master_df DataFrame
master_df = pd.merge(master_df, cpi_clean, on='Date', how='left')

print(master_df.head())

        Date    CPI
0 1978-01-01  64.30
1 1978-02-01  64.50
2 1978-03-01  64.70
3 1978-04-01  65.25
4 1978-05-01  65.80
        Date  House_Index     CPI
0 1984-01-01        57.37  102.90
1 1984-04-01        59.74  103.40
2 1984-07-01        60.50  104.40
3 1984-10-01        60.87  106.35
4 1985-01-01        61.61  106.60


In [91]:
# Load your data
Poverty_Rate = pd.read_csv(os.path.join('New_Data', 'Poverty_Rate.csv'))

# Convert Year to datetime (Jan 1st of each year)
Poverty_Rate['Date'] = pd.to_datetime(Poverty_Rate['Year'].astype(str) + '-01-01')
Poverty_Rate = Poverty_Rate[['Date', 'Poverty Rate (%)']].sort_values('Date')

# Set Date as index
Poverty_Rate.set_index('Date', inplace=True)

# Resample to quarterly start and interpolate
quarterly_df = Poverty_Rate.resample('QS').interpolate(method='linear')

# Reset index so Date is a column again
quarterly_df.reset_index(inplace=True)

print(quarterly_df.head(10))

quarterly_df.rename(columns={'Poverty Rate (%)': 'Poverty_Rate'}, inplace=True)

# Merge the Poverty Rate data with the master_df DataFrame
master_df = pd.merge(master_df, quarterly_df, on='Date', how='left')

print(master_df.head(60))

        Date  Poverty Rate (%)
0 1990-01-01            13.500
1 1990-04-01            13.675
2 1990-07-01            13.850
3 1990-10-01            14.025
4 1991-01-01            14.200
5 1991-04-01            14.350
6 1991-07-01            14.500
7 1991-10-01            14.650
8 1992-01-01            14.800
9 1992-04-01            14.875
         Date  House_Index     CPI  Poverty_Rate
0  1984-01-01        57.37  102.90           NaN
1  1984-04-01        59.74  103.40           NaN
2  1984-07-01        60.50  104.40           NaN
3  1984-10-01        60.87  106.35           NaN
4  1985-01-01        61.61  106.60           NaN
5  1985-04-01        62.97  108.20           NaN
6  1985-07-01        64.41  109.50           NaN
7  1985-10-01        64.83  110.15           NaN
8  1986-01-01        65.74  112.20           NaN
9  1986-04-01        67.53  111.55           NaN
10 1986-07-01        69.51  111.50           NaN
11 1986-10-01        71.18  112.85           NaN
12 1987-01-01        7

In [92]:
Unemployment_Rate = pd.read_csv(os.path.join('New_Data', 'DC_Unemployment_Rate.csv'))

# Convert date to datetime
Unemployment_Rate['Label'] = pd.to_datetime(Unemployment_Rate['Label'])
# Rename Label to Date
Unemployment_Rate.rename(columns={'Label': 'Date'}, inplace=True)
# Convert Unemployment Rate to float
Unemployment_Rate['Value'] = Unemployment_Rate['Value'].astype(float)
# Rename Value column to Unemployment_Rate
Unemployment_Rate.rename(columns={'Value': 'Unemployment_Rate'}, inplace=True)

print(Unemployment_Rate.head())

# Merge the Unemployment Rate data with the master_df DataFrame
master_df = pd.merge(master_df, Unemployment_Rate, on='Date', how='left')
print(master_df.head(40))




        Date  Unemployment_Rate
0 1990-01-01                2.7
1 1990-02-01                2.8
2 1990-03-01                2.6
3 1990-04-01                2.6
4 1990-05-01                2.8
         Date  House_Index     CPI  Poverty_Rate  Unemployment_Rate
0  1984-01-01        57.37  102.90           NaN                NaN
1  1984-04-01        59.74  103.40           NaN                NaN
2  1984-07-01        60.50  104.40           NaN                NaN
3  1984-10-01        60.87  106.35           NaN                NaN
4  1985-01-01        61.61  106.60           NaN                NaN
5  1985-04-01        62.97  108.20           NaN                NaN
6  1985-07-01        64.41  109.50           NaN                NaN
7  1985-10-01        64.83  110.15           NaN                NaN
8  1986-01-01        65.74  112.20           NaN                NaN
9  1986-04-01        67.53  111.55           NaN                NaN
10 1986-07-01        69.51  111.50           NaN            

In [93]:
Median_Household_Income = pd.read_csv(os.path.join('New_Data', 'DC_Median_Household_Income_Annually.csv'))

# Convert observation_date to datetime
Median_Household_Income['observation_date'] = pd.to_datetime(Median_Household_Income['observation_date'])
# Rename observation_date to Date
Median_Household_Income.rename(columns={'observation_date': 'Date'}, inplace=True)

# Set date as index
Median_Household_Income.set_index('Date', inplace=True)

# Resample to quarterly start and interpolate
quarterly_income_df = Median_Household_Income.resample('QS').interpolate(method='linear')

# Reset index so Date is a column again
quarterly_income_df.reset_index(inplace=True)

print(quarterly_income_df.head(10))

# Rename the column to Median_Household_Income
quarterly_income_df.rename(columns={'MEHOINUSDCA672N': 'Median_Household_Income'}, inplace=True)

# Merge the Median Household Income data with the master_df DataFrame
master_df = pd.merge(master_df, quarterly_income_df, on='Date', how='left')

print(master_df.head())


        Date  MEHOINUSDCA672N
0 1984-01-01          53650.0
1 1984-04-01          53635.0
2 1984-07-01          53620.0
3 1984-10-01          53605.0
4 1985-01-01          53590.0
5 1985-04-01          55402.5
6 1985-07-01          57215.0
7 1985-10-01          59027.5
8 1986-01-01          60840.0
9 1986-04-01          62237.5
        Date  House_Index     CPI  Poverty_Rate  Unemployment_Rate  \
0 1984-01-01        57.37  102.90           NaN                NaN   
1 1984-04-01        59.74  103.40           NaN                NaN   
2 1984-07-01        60.50  104.40           NaN                NaN   
3 1984-10-01        60.87  106.35           NaN                NaN   
4 1985-01-01        61.61  106.60           NaN                NaN   

   Median_Household_Income  
0                  53650.0  
1                  53635.0  
2                  53620.0  
3                  53605.0  
4                  53590.0  


In [94]:
# Load the DC Population data
Population = pd.read_csv(os.path.join('New_Data', 'DC_Population.csv'))

# Convert observation_date to datetime
Population['observation_date'] = pd.to_datetime(Population['observation_date'])

# Rename WSHPOP to Population
Population.rename(columns={'observation_date': 'Date', 'WSHPOP': 'Population'}, inplace=True)

# Set date as index
Population.set_index('Date', inplace=True)

# Resample to quarterly start and interpolate
quarterly_population_df = Population.resample('QS').interpolate(method='linear')

# Reset index so Date is a column again
quarterly_population_df.reset_index(inplace=True)

print(quarterly_population_df.head(10))

# Create the row 1990-01-01, take the difference between the first two rows and subtract to get the population for 1990-01-01
new_row = pd.DataFrame({'Date': [pd.to_datetime('1990-01-01')], 'Population': [quarterly_population_df['Population'].iloc[0] - (quarterly_population_df['Population'].iloc[1] - quarterly_population_df['Population'].iloc[0])]})

quarterly_population_df = pd.concat([new_row, quarterly_population_df], ignore_index=True)

print(quarterly_population_df.head(10))

# Merge the Population data with the master_df DataFrame
master_df = pd.merge(master_df, quarterly_population_df, on='Date', how='left')
print(master_df.head(40))

        Date   Population
0 1990-04-01  4105.955000
1 1990-07-01  4123.395878
2 1990-10-01  4140.836756
3 1991-01-01  4158.277634
4 1991-04-01  4175.718512
5 1991-07-01  4193.159390
6 1991-10-01  4210.600268
7 1992-01-01  4228.041146
8 1992-04-01  4245.482024
9 1992-07-01  4262.922902
        Date   Population
0 1990-01-01  4088.514122
1 1990-04-01  4105.955000
2 1990-07-01  4123.395878
3 1990-10-01  4140.836756
4 1991-01-01  4158.277634
5 1991-04-01  4175.718512
6 1991-07-01  4193.159390
7 1991-10-01  4210.600268
8 1992-01-01  4228.041146
9 1992-04-01  4245.482024
         Date  House_Index     CPI  Poverty_Rate  Unemployment_Rate  \
0  1984-01-01        57.37  102.90           NaN                NaN   
1  1984-04-01        59.74  103.40           NaN                NaN   
2  1984-07-01        60.50  104.40           NaN                NaN   
3  1984-10-01        60.87  106.35           NaN                NaN   
4  1985-01-01        61.61  106.60           NaN                NaN   
5 

In [95]:
Interest_Rate = pd.read_csv(os.path.join('New_Data', 'Interest_Rates.csv'))

# Convert observation_date to datetime
Interest_Rate['observation_date'] = pd.to_datetime(Interest_Rate['observation_date'])
# Rename observation_date to Date
Interest_Rate.rename(columns={'observation_date': 'Date'}, inplace=True)
# Convert FEDFUNDS to float
Interest_Rate['FEDFUNDS'] = Interest_Rate['FEDFUNDS'].astype(float)
# Rename FEDFUNDS column to Interest_Rate
Interest_Rate.rename(columns={'FEDFUNDS': 'Interest_Rate'}, inplace=True)

# Merge the Interest Rate data with the master_df DataFrame
master_df = pd.merge(master_df, Interest_Rate, on='Date', how='left')

print(master_df.head())

        Date  House_Index     CPI  Poverty_Rate  Unemployment_Rate  \
0 1984-01-01        57.37  102.90           NaN                NaN   
1 1984-04-01        59.74  103.40           NaN                NaN   
2 1984-07-01        60.50  104.40           NaN                NaN   
3 1984-10-01        60.87  106.35           NaN                NaN   
4 1985-01-01        61.61  106.60           NaN                NaN   

   Median_Household_Income  Population  Interest_Rate  
0                  53650.0         NaN           9.56  
1                  53635.0         NaN          10.29  
2                  53620.0         NaN          11.23  
3                  53605.0         NaN           9.99  
4                  53590.0         NaN           8.35  


In [96]:
# Load data
Mortgage_Rate = pd.read_csv(os.path.join('New_Data', 'Mortgage_Rate.csv'))

# Convert observation_date to datetime and rename
Mortgage_Rate['observation_date'] = pd.to_datetime(Mortgage_Rate['observation_date'])
Mortgage_Rate.rename(columns={'observation_date': 'Date', 'MORTGAGE30US': 'Mortgage_Rate'}, inplace=True)

# Convert to float, if needed
Mortgage_Rate['Mortgage_Rate'] = Mortgage_Rate['Mortgage_Rate'].astype(float)

# Set Date as index to resample
Mortgage_Rate.set_index('Date', inplace=True)

# Resample to monthly start (or use 'Q' for quarterly average)
Mortgage_Rate = Mortgage_Rate.resample('MS').mean().reset_index()

# Merge with master_df on Date
master_df = pd.merge(master_df, Mortgage_Rate, on='Date', how='left')

print(master_df.head())

        Date  House_Index     CPI  Poverty_Rate  Unemployment_Rate  \
0 1984-01-01        57.37  102.90           NaN                NaN   
1 1984-04-01        59.74  103.40           NaN                NaN   
2 1984-07-01        60.50  104.40           NaN                NaN   
3 1984-10-01        60.87  106.35           NaN                NaN   
4 1985-01-01        61.61  106.60           NaN                NaN   

   Median_Household_Income  Population  Interest_Rate  Mortgage_Rate  
0                  53650.0         NaN           9.56        13.3675  
1                  53635.0         NaN          10.29        13.6525  
2                  53620.0         NaN          11.23        14.6675  
3                  53605.0         NaN           9.99        14.1300  
4                  53590.0         NaN           8.35        13.0750  


In [97]:
Gross_Domestic_Product = pd.read_csv(os.path.join('New_Data', 'GDP_growth.csv'))

# Convert observation_date to datetime
Gross_Domestic_Product['observation_date'] = pd.to_datetime(Gross_Domestic_Product['observation_date'])
# Rename observation_date to Date
Gross_Domestic_Product.rename(columns={'observation_date': 'Date'}, inplace=True)
# Convert GDP to float
Gross_Domestic_Product['GDP'] = Gross_Domestic_Product['GDP'].astype(float)

# Merge the GDP data with the master_df DataFrame
master_df = pd.merge(master_df, Gross_Domestic_Product, on='Date', how='left')

print(master_df.head())


        Date  House_Index     CPI  Poverty_Rate  Unemployment_Rate  \
0 1984-01-01        57.37  102.90           NaN                NaN   
1 1984-04-01        59.74  103.40           NaN                NaN   
2 1984-07-01        60.50  104.40           NaN                NaN   
3 1984-10-01        60.87  106.35           NaN                NaN   
4 1985-01-01        61.61  106.60           NaN                NaN   

   Median_Household_Income  Population  Interest_Rate  Mortgage_Rate       GDP  
0                  53650.0         NaN           9.56        13.3675  3908.054  
1                  53635.0         NaN          10.29        13.6525  4009.601  
2                  53620.0         NaN          11.23        14.6675  4084.250  
3                  53605.0         NaN           9.99        14.1300  4148.551  
4                  53590.0         NaN           8.35        13.0750  4230.168  


In [99]:
# Save the DataFrame to a CSV file
output_file = 'DC_Master_Data.csv'
master_df.to_csv(output_file, index=False)